In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pathlib import Path
from typing import Dict, List, Optional

In [2]:
RESULTS_DIR = Path("results")
PLOTS_DIR   = RESULTS_DIR / "plots"
LOGS_DIR    = RESULTS_DIR / "logs"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

COLORS = {
    "fedavg": "#4C72B0",
    "efl": "#DD8452",
    "no_defense": "#C44E52",
    "gaussian": "#55A868",
    "subj_aware": "#8172B2",
    "dp_sgd": "#937860",
}

In [4]:
def plot_training_curves(
    histories:  Dict[str, List[dict]],   # {"FedAvg": [...], "EFL": [...]}
    metric:     str = "act_acc",
    title:      str = "Activity recognition accuracy over FL rounds",
    save_name:  Optional[str] = "training_curves.png",
) -> plt.Figure:
    """
    Plot one line per experiment across training rounds.
    histories: {label: list_of_round_dicts}
    """
    fig, ax = plt.subplots(figsize=(8, 4.5))

    for label, history in histories.items():
        rounds = [d["round"]   for d in history]
        vals   = [d[metric]    for d in history]
        color  = COLORS.get(label.lower().replace(" ", "_"), None)
        ax.plot(rounds, vals, marker="o", markersize=3,
                label=label, color=color, linewidth=1.8)

    ax.set_xlabel("FL round")
    y_label = "Activity accuracy" if metric == "act_acc" else "Subject accuracy (ASR)"
    ax.set_ylabel(y_label)
    ax.set_title(title, pad=10)
    ax.legend(framealpha=0.7)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    plt.tight_layout()

    if save_name:
        path = PLOTS_DIR / save_name
        fig.savefig(path, bbox_inches="tight")
        print(f"[eval] Saved → {path}")

    return fig


In [5]:
def plot_privacy_utility(
    results:    List[Dict],   # list of dicts from defenses.evaluate_defense()
    group_by:   str = "defense",
    save_name:  Optional[str] = "privacy_utility.png",
) -> plt.Figure:
    """
    Scatter plot: x = activity accuracy (utility), y = 1 - ASR (privacy).
    Each point is one (defense, noise_level) config.
    Pareto-optimal configs will appear in the top-right.
    """
    df  = pd.DataFrame(results)
    df["privacy"] = 1.0 - df["asr"]

    fig, ax = plt.subplots(figsize=(7, 5))

    for defense, grp in df.groupby(group_by):
        color = COLORS.get(defense.lower().replace(" ", "_"), None)
        ax.scatter(grp["act_acc"], grp["privacy"],
                   label=defense, color=color, s=60, zorder=3)
        # connect dots in order of noise level
        grp_sorted = grp.sort_values("noise_level")
        ax.plot(grp_sorted["act_acc"], grp_sorted["privacy"],
                color=color, alpha=0.4, linewidth=1)

    # baseline reference (no defense)
    if "baseline_asr" in df.columns:
        bl_asr = df["baseline_asr"].iloc[0]
        ax.axhline(1 - bl_asr, color="gray", linestyle="--",
                   linewidth=1, alpha=0.6, label="Baseline (no defense)")

    ax.set_xlabel("Activity accuracy (utility ↑)")
    ax.set_ylabel("1 – ASR (privacy ↑)")
    ax.set_title("Privacy–utility tradeoff", pad=10)
    ax.legend(framealpha=0.7)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

    # ideal point marker
    ax.annotate("ideal", xy=(0.99, 0.99), fontsize=9, color="green",
                ha="right", style="italic")
    ax.annotate("↗", xy=(0.97, 0.97), fontsize=14, color="green")

    plt.tight_layout()

    if save_name:
        path = PLOTS_DIR / save_name
        fig.savefig(path, bbox_inches="tight")
        print(f"[eval] Saved → {path}")

    return fig


In [6]:
def plot_attack_comparison(
    results:   Dict[str, float],   # {"No defense": 0.82, "Gaussian σ=0.1": 0.61, ...}
    baseline:  float = None,
    title:     str   = "Attack success rate by configuration",
    save_name: Optional[str] = "attack_comparison.png",
) -> plt.Figure:
    """
    Horizontal bar chart of ASR per configuration.
    Lower is better (more private).
    """
    labels = list(results.keys())
    values = list(results.values())

    fig, ax = plt.subplots(figsize=(8, 0.6 * len(labels) + 2))
    colors  = [COLORS.get(l.lower().split()[0], "#6c757d") for l in labels]
    bars    = ax.barh(labels, values, color=colors, height=0.5, edgecolor="white")

    if baseline is not None:
        ax.axvline(baseline, color="gray", linestyle="--",
                   linewidth=1.2, label=f"Baseline ASR = {baseline:.2f}")
        ax.legend(fontsize=9)

    ax.set_xlabel("Attack success rate (lower = more private)")
    ax.set_title(title, pad=10)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.invert_yaxis()

    for bar, val in zip(bars, values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.2%}", va="center", fontsize=9)

    plt.tight_layout()

    if save_name:
        path = PLOTS_DIR / save_name
        fig.savefig(path, bbox_inches="tight")
        print(f"[eval] Saved → {path}")

    return fig

In [7]:
def plot_confusion_matrix(
    y_true:    np.ndarray,
    y_pred:    np.ndarray,
    class_names: Optional[List[str]] = None,
    title:     str = "Activity confusion matrix",
    save_name: Optional[str] = "confusion_matrix.png",
) -> plt.Figure:
    from sklearn.metrics import confusion_matrix as cm
    mat = cm(y_true, y_pred, normalize="true")

    if class_names is None:
        class_names = [f"Class {i}" for i in range(mat.shape[0])]

    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=10)
    plt.tight_layout()

    if save_name:
        path = PLOTS_DIR / save_name
        fig.savefig(path, bbox_inches="tight")
        print(f"[eval] Saved → {path}")

    return fig

In [8]:
def save_results_csv(
    results: List[Dict],
    filename: str = "results.csv",
) -> Path:
    df   = pd.DataFrame(results)
    path = LOGS_DIR / filename
    df.to_csv(path, index=False)
    print(f"[eval] Results saved → {path}")
    return path


In [10]:
def print_summary_table(results: List[Dict]) -> None:
    df = pd.DataFrame(results)
    cols = ["defense", "noise_level", "act_acc", "asr", "baseline_asr",
            "privacy_gain", "utility_cost", "tradeoff_score"]
    cols = [c for c in cols if c in df.columns]
    df   = df[cols].sort_values("tradeoff_score", ascending=False)

    print("\n" + "=" * 80)
    print("EXPERIMENT SUMMARY")
    print("=" * 80)
    print(df.to_string(index=False, float_format="{:.4f}".format))
    print("=" * 80)
    print("Higher tradeoff_score = better privacy–utility balance\n")
